In [1]:
import pandas as pd

# Load the training and test datasets
train_df = pd.read_csv('rating10user91_trainset.csv')
test_df = pd.read_csv('rating10user91_testset.csv')

print("Training data loaded successfully.")
print(train_df.head())
print("\nTest data loaded successfully.")
print(test_df.head())

Training data loaded successfully.
   userid       isbn  rating
0    6251   60392452      10
1    6251   61009059       7
2    6251  140067477      10
3    6251  375727345       6
4    6251  380789035       7

Test data loaded successfully.
   userid        isbn  rating
0    6251   312980140       7
1    6251   385484518       7
2    6251   439136350      10
3    6251  043935806X       9
4    6251   440206154       7


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

# --- Data Preparation ---
# Create a pivot table for user-item ratings
train_matrix = train_df.pivot_table(index='userid', columns='isbn', values='rating')

# --- User Similarity Calculation (Pearson Correlation) ---

# Function to calculate Pearson correlation between two users
def pearson_correlation(user_a_ratings, user_b_ratings):
    # Get common items rated by both users
    common_items = user_a_ratings.index.intersection(user_b_ratings.index)

    # Filter ratings for common items
    ratings_a = user_a_ratings.loc[common_items]
    ratings_b = user_b_ratings.loc[common_items]

    # If there are no common items, similarity is 0
    if ratings_a.empty or ratings_b.empty:
        return 0

    # Calculate the mean of ratings for common items for each user
    mean_a = np.mean(ratings_a)
    mean_b = np.mean(ratings_b)

    # Calculate the numerator (sum of product of differences from the mean)
    numerator = np.sum((ratings_a - mean_a) * (ratings_b - mean_b))

    # Calculate the denominator (product of standard deviations)
    std_dev_a = np.sqrt(np.sum((ratings_a - mean_a)**2))
    std_dev_b = np.sqrt(np.sum((ratings_b - mean_b)**2))

    denominator = std_dev_a * std_dev_b

    # Avoid division by zero
    if denominator == 0:
        return 0
    else:
        return numerator / denominator

# Create a DataFrame to store similarities
# Initialize with zeros
similarity_matrix = pd.DataFrame(index=train_matrix.index, columns=train_matrix.index, data=0.0)

# Iterate over all unique user pairs to calculate similarity
users = train_matrix.index
for i in range(len(users)):
    for j in range(i + 1, len(users)):
        user1 = users[i]
        user2 = users[j]

        # Get ratings for user1 and user2, ignoring NaNs for PCC
        # We need to be careful here. For PCC, we only consider items BOTH users have rated.
        # The pivot_table will have NaNs for items not rated.

        # Get the row for each user
        user1_ratings = train_matrix.loc[user1].dropna()
        user2_ratings = train_matrix.loc[user2].dropna()

        # Calculate similarity using only common items
        sim = pearson_correlation(user1_ratings, user2_ratings)

        similarity_matrix.loc[user1, user2] = sim
        similarity_matrix.loc[user2, user1] = sim # Similarity is symmetric

print("Similarity matrix calculated.")
print(similarity_matrix.head())

# --- Storing the results ---
# Save the similarity matrix to a CSV file
similarity_matrix.to_csv('P2Part1_1PCC_Group4.csv')

print(f"User similarities (Pearson Correlation Coefficient) saved to 'P2Part1_1PCC_Group4.csv'")


Similarity matrix calculated.
userid  6251      6575      7346      11676     13552     16795     17950   \
userid                                                                       
6251       0.0  0.000000  0.000000  0.000000  0.000000  1.000000  0.000000   
6575       0.0  0.000000  0.000000  0.228692  0.000000  0.000000  0.000000   
7346       0.0  0.000000  0.000000 -0.185419  0.000000 -1.000000  0.000000   
11676      0.0  0.228692 -0.185419  0.000000 -0.603023  0.095570 -0.991241   
13552      0.0  0.000000  0.000000 -0.603023  0.000000  0.394771  1.000000   

userid    21014     23872     23902   ...    241980    242083    245410  \
userid                                ...                                 
6251    0.000000  0.000000  0.000000  ...  0.000000  1.000000  1.000000   
6575    0.000000  0.000000 -1.000000  ...  0.000000  0.000000  0.000000   
7346    0.000000  0.000000  0.000000  ... -1.000000 -1.000000  0.000000   
11676  -0.413664  0.904534  0.209032  ... -0.065

In [4]:
import pandas as pd
import numpy as np

# --- Data Loading and Initial Checks ---
try:
    train_df = pd.read_csv('rating10user91_trainset.csv')
    test_df = pd.read_csv('rating10user91_testset.csv')
    print("Training and test CSV files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading CSV file: {e}. Make sure the files are in the correct directory.")
    exit()

# Load the similarity matrix
try:
    similarity_matrix = pd.read_csv('P2Part1_1PCC_Group4.csv', index_col='userid')
    print("Similarity matrix loaded.")
except FileNotFoundError:
    print("Error: 'P2Part1_1PCC_Group4.csv' not found. Please run the previous step first to generate it.")
    exit()

# --- Matrix Creation and Debugging ---
print("\n--- Debugging Matrix Creation ---")

# Convert userid columns to string type for consistent comparison
train_df['userid'] = train_df['userid'].astype(str)
test_df['userid'] = test_df['userid'].astype(str)

# Re-index similarity matrix with string user IDs if it's not already
similarity_matrix.index = similarity_matrix.index.astype(str)
similarity_matrix.columns = similarity_matrix.columns.astype(str)

# Create user-item matrix for training data
train_matrix = train_df.pivot_table(index='userid', columns='isbn', values='rating')
print(f"Shape of training matrix: {train_matrix.shape}")

# Create user-item matrix for test data
test_matrix = test_df.pivot_table(index='userid', columns='isbn', values='rating')
print(f"Shape of test matrix: {test_matrix.shape}")

# Get lists of user IDs for cross-referencing
users_in_train_df = train_df['userid'].unique().tolist()
users_in_test_df = test_df['userid'].unique().tolist()
users_in_train_matrix_index = train_matrix.index.tolist()
users_in_similarity_index = similarity_matrix.index.tolist()

print(f"Unique users in train_df: {len(users_in_train_df)}")
print(f"Unique users in test_df: {len(users_in_test_df)}")
print(f"Unique users in train_matrix index: {len(users_in_train_matrix_index)}")
print(f"Unique users in similarity_matrix index: {len(users_in_similarity_index)}")

# --- Identify potential issues ---
# Users in test_df but not in train_matrix index
users_only_in_test = set(users_in_test_df) - set(users_in_train_matrix_index)
if users_only_in_test:
    print(f"\nWARNING: Users found in test_df but NOT in train_matrix index: {list(users_only_in_test)[:5]}...")
else:
    print("\nAll users in test_df are also present in train_matrix index (or have no predictions generated for them).")

# Users in train_matrix index but not in similarity_matrix index
users_in_train_but_not_sim = set(users_in_train_matrix_index) - set(users_in_similarity_index)
if users_in_train_but_not_sim:
     print(f"WARNING: Users in train_matrix index but NOT in similarity_matrix index: {list(users_in_train_but_not_sim)[:5]}...")

print("--- End Debugging Matrix Creation ---\n")

# --- Prediction Function ---
def predict_ratings(target_user_id, test_user_item_matrix, train_user_item_matrix, similarity_matrix, k=5):
    predictions = {}

    # --- Check if the target user exists in the training data AND similarity matrix ---
    if target_user_id not in similarity_matrix.index:
        print(f"User {target_user_id} not found in similarity matrix index. Skipping predictions for this user.")
        return {}

    # Calculate target user's average rating from the training data
    try:
        # Ensure target_user_id is treated as string if matrix index is string
        target_user_avg_rating = np.mean(train_user_item_matrix.loc[target_user_id].dropna())
    except KeyError:
        print(f"Error: Could not retrieve average rating for user {target_user_id} from training matrix.")
        return {}

    # Get the items rated by the target user in the training set
    rated_items_by_target_user_in_train = train_user_item_matrix.loc[target_user_id].dropna().index.tolist()

    # Get the items present in the test set's columns that are also in the training data
    all_train_items = train_user_item_matrix.columns
    items_to_predict_for = [item for item in test_user_item_matrix.columns if item in all_train_items]

    for item_id in items_to_predict_for:
        if item_id not in rated_items_by_target_user_in_train:
            neighbor_data = []
            # Iterate through users in the similarity matrix columns (potential neighbors)
            for neighbor_id in similarity_matrix.columns:
                if neighbor_id == target_user_id:
                    continue

                # Check if the neighbor has rated this item in the training set
                if pd.notna(train_user_item_matrix.loc[neighbor_id, item_id]):
                    try:
                        neighbor_avg_rating = np.mean(train_user_item_matrix.loc[neighbor_id].dropna())
                        sim_value = similarity_matrix.loc[target_user_id, neighbor_id] # Access similarity

                        neighbor_data.append({
                            'neighbor_id': neighbor_id,
                            'rating': train_user_item_matrix.loc[neighbor_id, item_id],
                            'avg_rating': neighbor_avg_rating,
                            'similarity': sim_value
                        })
                    except KeyError:
                        print(f"KeyError accessing similarity for ({target_user_id}, {neighbor_id}) or training rating for {neighbor_id}.")
                        continue
                    except Exception as e:
                        print(f"An error occurred processing neighbor {neighbor_id} for item {item_id}: {e}")
                        continue

            if neighbor_data:
                neighbor_df = pd.DataFrame(neighbor_data)
                top_k_neighbors = neighbor_df.nlargest(k, 'similarity')

                weighted_sum_numerator = 0
                sum_of_similarities = 0

                for _, row in top_k_neighbors.iterrows():
                    neighbor_sim = row['similarity']
                    neighbor_rating = row['rating']
                    neighbor_avg = row['avg_rating']
                    weighted_sum_numerator += neighbor_sim * (neighbor_rating - neighbor_avg)
                    sum_of_similarities += neighbor_sim

                if sum_of_similarities != 0:
                    predicted_rating = target_user_avg_rating + (weighted_sum_numerator / sum_of_similarities)
                    predicted_rating = max(1, min(10, predicted_rating))
                    predictions[(target_user_id, item_id)] = predicted_rating
                else:
                    predictions[(target_user_id, item_id)] = target_user_avg_rating
            else:
                predictions[(target_user_id, item_id)] = target_user_avg_rating

    return predictions

# --- Generate Predictions ---
all_predictions = {}
k_value = 5

test_users = test_df['userid'].unique() # Already converted to string type above

for user_id in test_users:
    # This check is now more robust as we ensure user_id is string and check against similarity_matrix index (also string)
    if user_id in similarity_matrix.index: # Check if user_id (string) is in the index (list of strings)
        print(f"Predicting ratings for user: {user_id}")
        user_predictions = predict_ratings(
            user_id,
            test_matrix,
            train_matrix,
            similarity_matrix,
            k=k_value
        )
        all_predictions.update(user_predictions)
    else:
        print(f"User {user_id} not found in similarity matrix index. Skipping predictions for this user.")

# --- Format and Save Predictions ---
predictions_list = []
for (user, item), rating in all_predictions.items():
    predictions_list.append({'userid': user, 'isbn': item, 'predicted_rating': rating})

predicted_ratings_df = pd.DataFrame(predictions_list)

if not predicted_ratings_df.empty:
    predicted_ratings_df.to_csv('P2Part1_1PCC_Group4_predictions.csv', index=False)
    print("\nPredicted ratings for unseen items saved to 'P2Part1_1PCC_Group4_predictions.csv'")
    print(predicted_ratings_df.head())
else:
    print("\nNo predictions were generated.")


Training and test CSV files loaded successfully.
Similarity matrix loaded.

--- Debugging Matrix Creation ---
Shape of training matrix: (91, 112)
Shape of test matrix: (40, 91)
Unique users in train_df: 91
Unique users in test_df: 40
Unique users in train_matrix index: 91
Unique users in similarity_matrix index: 91

All users in test_df are also present in train_matrix index (or have no predictions generated for them).
--- End Debugging Matrix Creation ---

Predicting ratings for user: 6251
Predicting ratings for user: 6575
Predicting ratings for user: 7346
Predicting ratings for user: 11676
Predicting ratings for user: 13552
Predicting ratings for user: 16795
Predicting ratings for user: 17950
Predicting ratings for user: 21014
Predicting ratings for user: 23872
Predicting ratings for user: 23902
Predicting ratings for user: 28634
Predicting ratings for user: 30735
Predicting ratings for user: 35857
Predicting ratings for user: 35859
Predicting ratings for user: 37712
Predicting ratin

In [7]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load the data
df = pd.read_csv('P2Part1_1PCC_Group4_predictions.csv')

# Pivot to create user-item matrix
user_item_matrix = df.pivot_table(index='userid', columns='isbn', values='predicted_rating', fill_value=0)

# Compute cosine similarity between users
similarity_matrix = pd.DataFrame(cosine_similarity(user_item_matrix), 
                                 index=user_item_matrix.index, 
                                 columns=user_item_matrix.index)

# Function to get top 5 neighbors for a user
def get_top_neighbors(user_id, sim_matrix, top_n=5):
    neighbors = sim_matrix.loc[user_id].drop(user_id).sort_values(ascending=False).head(top_n)
    return neighbors.index.tolist()

# Prepare final output
output_rows = []

for user_id in df['userid'].unique():
    neighbors = get_top_neighbors(user_id, similarity_matrix)
    top_books = df[df['userid'] == user_id].sort_values(by='predicted_rating', ascending=False).head(5)
    
    for _, row in top_books.iterrows():
        output_rows.append({
            'TargetUserID': user_id,
            '1stNNUserID': neighbors[0],
            '2NNUserID': neighbors[1],
            '3NNUserID': neighbors[2],
            '4NNUserID': neighbors[3],
            '5NNUserID': neighbors[4],
            'Book’s ISBN': row['isbn'],
            'predicted rating': row['predicted_rating']
        })

# Convert to DataFrame
final_df = pd.DataFrame(output_rows)
final_df.to_csv('user_summary_top_books_and_neighbors.csv', index=False)
print(final_df)


     TargetUserID  1stNNUserID  2NNUserID  3NNUserID  4NNUserID  5NNUserID  \
0            6251        53174      52584      56447      23872      88693   
1            6251        53174      52584      56447      23872      88693   
2            6251        53174      52584      56447      23872      88693   
3            6251        53174      52584      56447      23872      88693   
4            6251        53174      52584      56447      23872      88693   
..            ...          ...        ...        ...        ...        ...   
195        107784        23902      88693      68555      93047      37950   
196        107784        23902      88693      68555      93047      37950   
197        107784        23902      88693      68555      93047      37950   
198        107784        23902      88693      68555      93047      37950   
199        107784        23902      88693      68555      93047      37950   

    Book’s ISBN  predicted rating  
0    080410526X         10.

In [10]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load the data
df = pd.read_csv('P2Part1_1PCC_Group4_predictions.csv')

# Create user-item matrix
user_item_matrix = df.pivot_table(index='userid', columns='isbn', values='predicted_rating', fill_value=0)

# Compute cosine similarity
similarity_matrix = pd.DataFrame(cosine_similarity(user_item_matrix),
                                 index=user_item_matrix.index,
                                 columns=user_item_matrix.index)

# Function to get top 5 neighbors
def get_top_neighbors(user_id, sim_matrix, top_n=5):
    neighbors = sim_matrix.loc[user_id].drop(user_id).sort_values(ascending=False).head(top_n)
    return neighbors.index.tolist()

# Build one row per user
output_rows = []

for user_id in df['userid'].unique():
    neighbors = get_top_neighbors(user_id, similarity_matrix)
    top_books = df[df['userid'] == user_id].sort_values(by='predicted_rating', ascending=False).head(5)
    
    # Format books as ISBN:Rating
    book_summary = ', '.join([f"{row['isbn']}:{round(row['predicted_rating'], 2)}" for _, row in top_books.iterrows()])
    
    output_rows.append({
        'TargetUserID': user_id,
        '1stNNUserID': neighbors[0],
        '2ndNNUserID': neighbors[1],
        '3rdNNUserID': neighbors[2],
        '4thNNUserID': neighbors[3],
        '5thNNUserID': neighbors[4],
        'TopBooks (ISBN:Rating)': book_summary
    })

# Final DataFrame
final_df = pd.DataFrame(output_rows)
final_df.to_csv('‘P2Part1_2Recommendation_Group4.csv',index=False)

PermissionError: [Errno 13] Permission denied: '‘P2Part1_2Recommendation_Group [group_no].csv'